# 第 3 章：一个 token 怎样“看”前面的 token？

`Q` 是当前 token 提出的问题，`K` 用来与问题匹配，`V` 是最后取回的信息。
流程：`Q @ Kᵀ → scores → softmax → weights → weights @ V → context`。
用 `B` 表示 batch 大小，`T` 表示 token 数，`D` 表示向量宽度。


In [ ]:
from pathlib import Path
import sys

# notebook 在项目根目录或章节目录中打开时，都能找到我们的 Python 文件。
ROOT = Path.cwd()
if not (ROOT / "chapter02").exists():
    ROOT = ROOT.parent
if not (ROOT / "chapter02").exists():
    raise FileNotFoundError("请从本项目的根目录或章节目录启动 Jupyter")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目目录:", ROOT)


## 1. 简化版：把输入直接当 Q、K、V

没有 `W_Q/W_K/W_V` 三张可训练表，但依然能根据输入向量算出注意力。
分数是点积：每个 token 向量与句中每个 token 向量比较。
分数高，经过 softmax 通常会得到更大权重。


In [ ]:
import torch
from chapter03.attention import (
    simplified_self_attention, SelfAttention_v1, SelfAttention_v2,
    CausalAttention, MultiHeadAttentionWrapper, MultiHeadAttention
)

torch.manual_seed(123)
x = torch.tensor([
    [0.43, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.64],
    [0.22, 0.58, 0.33],
    [0.77, 0.25, 0.10],
    [0.05, 0.80, 0.55],
])
z, weights = simplified_self_attention(x)
print("输入 [T,D]:", x.shape)
print("权重 [T,T]:", weights.shape)
print("每一行权重的和:", weights.sum(dim=-1))
print("上下文向量 [T,D]:", z.shape)


## 2. 可训练版本：用三张表学会生成 Q/K/V

v1 用 `nn.Parameter`，所以写 `x @ W`；v2 用 `nn.Linear`，所以写 `W(x)`。
`nn.Linear` 本身是模块，不是原始矩阵。它内部的 `.weight` 才是矩阵。
在 v1/v2 中，`x` 是一整组 token `[T,D]`；因果版本会增加 batch 维。


In [ ]:
v1 = SelfAttention_v1(d_in=3, d_out=4)
v2 = SelfAttention_v2(d_in=3, d_out=4)
print("v1 权重形状 [输入,输出]:", v1.W_query.shape)
print("v2 Linear.weight 形状 [输出,输入]:", v2.W_query.weight.shape)
print("v1 输出:", v1(x).shape, "v2 输出:", v2(x).shape)

q = v2.W_query(x)
k = v2.W_key(x)
print("Q:", q.shape, "K 转置:", k.T.shape, "scores:", (q @ k.T).shape)
print("√key_dim =", k.shape[-1] ** 0.5)


## 3. 因果注意力：看当前和以前，不能看未来

`[B,T,T]` 的分数表中，右上角是未来位置。mask 把那里的 score 设为负无穷；
softmax 后这些位置的权重为 0，于是不会进入当前 token 的 context。
`register_buffer` 保存这个固定 mask，但它不是可训练参数。dropout 只在 `train()` 模式生效。


In [ ]:
batch = x.unsqueeze(0).repeat(2, 1, 1)  # [B=2,T=6,D=3]
causal = CausalAttention(3, 4, context_length=6, dropout=0.0)
q, k = causal.W_query(batch), causal.W_key(batch)
scores = q @ k.transpose(1, 2)  # batch 中不能用 k.T，后者会翻转全部维度
masked_scores = scores.masked_fill(causal.mask[:6, :6], -torch.inf)
causal_weights = torch.softmax(masked_scores / k.shape[-1] ** 0.5, dim=-1)
print("mask (1 代表未来):\n", causal.mask.int())
print("第一句话的注意力权重:\n", causal_weights[0].round(decimals=3))
print("输出 [B,T,d_out]:", causal(batch).shape)


## 4. 多头：同一份输入，几种不同的关注方式

`MultiHeadAttentionWrapper` 建造多个独立小头；`MultiHeadAttention` 用大矩阵一次算完再分头。
在后一种写法中，先得到 `[B,T,d_out]`，再 `view` 为 `[B,T,H,Dh]`，其中 `d_out=H×Dh`。
**每个头都读完整的原输入**。`view` 只是把投影后的输出列分成几组，不是删去输入 token。


In [ ]:
wrapper = MultiHeadAttentionWrapper(
    d_in=3, d_out=2, context_length=6, dropout=0.0, num_heads=2
)
multi = MultiHeadAttention(
    d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2
)
print("独立小头的输出 [B,T,2×2]:", wrapper(batch).shape)
print("大矩阵分头的输出 [B,T,4]:", multi(batch).shape)
print("头数:", multi.num_heads, "每头输出宽度:", multi.head_dim)
print("大 Q 表形状:", multi.W_query.weight.shape)


`out_proj` 不负责拼接。拼接发生在 `transpose(...).contiguous().view(...)`。
`out_proj` 是拼接**之后**的线性层，学习如何混合不同头的结果。

对照自己手打代码，常见拼写错误有 `context_lenght`、`num_token`、`maks_bool`、`ann_scores`。
分头后 K 的形状是 `[B,H,T,Dh]`，计算 scores 要使用 `keys.transpose(2,3)`。
